In [1]:
import pandas as pd
from pycaret.classification import *
import catboost
import numpy as np
from sklearn.metrics import roc_curve, auc, roc_auc_score


## 0. Import dataset

In [ ]:
dat = pd.read_csv('.\\datasets\\integrate_20.csv')
dat_org = dat.copy()

In [3]:
dat.head()

,sPD_M,sPD_SD,sPD_H_PD,label,case,sPAS_M,sPAS_SD,sPD_M_diff,sPD_H_PD_diff,sPD_CS,sPAS_M_diff,sPAS_CS
0,0.499130,0.423468,0.000375,Competition,f1,0.007780,0.553071,-0.844863,-0.006877,0.719269,-0.022563,-0.291352
1,0.506910,0.429594,0.000032,Competition,f1,0.044840,0.556272,-0.867427,-0.006479,0.731844,0.058042,-0.265908
2,0.551751,0.467941,0.000208,Competition,f1,0.020543,0.566928,-0.809385,-0.008703,0.739015,-0.047605,-0.394532
3,0.572293,0.473938,0.000422,Competition,f1,0.047020,0.530554,-0.856990,-0.005920,0.758199,0.003676,-0.270335
4,0.619313,0.408232,0.000074,Competition,f1,0.012094,0.509675,-0.853314,-0.006674,0.799156,-0.070278,-0.145437


## 1. Task setting

In [ ]:
pos = 'Intimacy' 
neg = 'non_Intimacy'
dat = dat_org.loc[dat_org.label.isin([pos, neg])].reset_index(drop=True)
dat['case'] = [1 if ('f' in i) else 0 for i in dat['case']]

train = dat

## 2. Pycaret pipeline setting

In [5]:
reg_test_1 = setup(data=train,
                   target='label',
                   index=False,
                   session_id = 1,
                   train_size= 0.9,
                   use_gpu=True
                   )

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Number of positive: 1, number of negative: 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that mee

,Description,Value
0,Session id,1
1,Target,label
2,Target type,Binary
3,Target mapping,"Intimacy: 0, non_Intimacy: 1"
4,Original data shape,"(7780, 12)"
5,Transformed data shape,"(7780, 12)"
6,Transformed train set shape,"(7002, 12)"
7,Transformed test set shape,"(778, 12)"
8,Numeric features,11
9,Preprocess,True


[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Number of positive: 1, number of negative: 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that mee

## 3. Start training

In [8]:
model_lgbm = create_model('lightgbm') 

tuned_lgbm = tune_model(model_lgbm, optimize='Accuracy', n_iter=30, verbose = False)
final_lgbm = finalize_model(tuned_lgbm)
result2 = pull()

gain_importance1 = final_lgbm.feature_importances_


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9173,0.9739,0.9173,0.9179,0.9171,0.8336,0.8345
1,0.9073,0.9695,0.9073,0.9082,0.9071,0.8134,0.8146
2,0.9171,0.9747,0.9171,0.9175,0.9172,0.8340,0.8342
3,0.9243,0.9779,0.9243,0.9243,0.9243,0.8481,0.8481
4,0.9057,0.9724,0.9057,0.9060,0.9056,0.8104,0.8109
5,0.9114,0.9743,0.9114,0.9114,0.9114,0.8223,0.8223
6,0.9357,0.9857,0.9357,0.9358,0.9357,0.8709,0.8710
7,0.9100,0.9696,0.9100,0.9101,0.9099,0.8192,0.8195
8,0.9157,0.9789,0.9157,0.9158,0.9157,0.8307,0.8309


[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 2976, number of negative: 3325
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2552
[LightGBM] [Info] Number of data points in the train set: 6301, number of used features: 11
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3090, Ve

In [144]:
model_cat = create_model('catboost') 
model_lgbm = create_model('lightgbm') 
model_rf = create_model('rf') 

tuned_cat = tune_model(model_cat, optimize='Accuracy', n_iter=30, verbose = False)
final_cat = finalize_model(tuned_cat)
result1 = pull()

tuned_lgbm = tune_model(model_lgbm, optimize='Accuracy', n_iter=30, verbose = False)
final_lgbm = finalize_model(tuned_lgbm)
result2 = pull()

tuned_rf = tune_model(model_rf, optimize='Accuracy', n_iter=30, verbose = False)
final_rf = finalize_model(tuned_rf)
result3 = pull()

result= pd.concat([result1.iloc[10], result2.iloc[10], result3.iloc[10]], axis = 1).T
print(result)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9158,0.9711,0.9158,0.9162,0.9157,0.8308,0.8314
1,0.9258,0.9746,0.9258,0.9270,0.9256,0.8507,0.8521
2,0.9114,0.9745,0.9114,0.9118,0.9115,0.8225,0.8227
3,0.9314,0.9765,0.9314,0.9314,0.9314,0.8624,0.8624
4,0.9043,0.9717,0.9043,0.9043,0.9042,0.8077,0.8079
5,0.9257,0.9746,0.9257,0.9257,0.9257,0.8509,0.8510
6,0.9414,0.9856,0.9414,0.9414,0.9414,0.8825,0.8825
7,0.9071,0.9716,0.9071,0.9072,0.9071,0.8135,0.8137
8,0.9157,0.9765,0.9157,0.9157,0.9157,0.8309,0.8309


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9173,0.9739,0.9173,0.9179,0.9171,0.8336,0.8345
1,0.9073,0.9694,0.9073,0.9082,0.9071,0.8134,0.8146
2,0.9171,0.9746,0.9171,0.9175,0.9172,0.8340,0.8342
3,0.9243,0.9779,0.9243,0.9243,0.9243,0.8481,0.8481
4,0.9057,0.9724,0.9057,0.9060,0.9056,0.8104,0.8109
5,0.9114,0.9743,0.9114,0.9114,0.9114,0.8223,0.8223
6,0.9357,0.9858,0.9357,0.9358,0.9357,0.8709,0.8710
7,0.9100,0.9696,0.9100,0.9101,0.9099,0.8192,0.8195
8,0.9157,0.9790,0.9157,0.9158,0.9157,0.8307,0.8309


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9301,0.9812,0.9301,0.9303,0.9300,0.8596,0.8599
1,0.9258,0.9846,0.9258,0.9262,0.9257,0.8509,0.8514
2,0.9429,0.9880,0.9429,0.9430,0.9429,0.8854,0.8855
3,0.9443,0.9892,0.9443,0.9443,0.9443,0.8883,0.8883
4,0.9186,0.9802,0.9186,0.9191,0.9184,0.8362,0.8370
5,0.9357,0.9838,0.9357,0.9359,0.9357,0.8711,0.8712
6,0.9343,0.9873,0.9343,0.9344,0.9342,0.8680,0.8682
7,0.9214,0.9786,0.9214,0.9215,0.9214,0.8422,0.8424
8,0.9500,0.9883,0.9500,0.9500,0.9500,0.8997,0.8997


[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 2976, number of negative: 3325
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2552
[LightGBM] [Info] Number of data points in the train set: 6301, number of used features: 11
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Ve

In [153]:
tpr_cat = predict_model(tuned_cat, raw_score=True)
tpr_lgbm = predict_model(tuned_lgbm, raw_score=True)
tpr_rf = predict_model(tuned_rf, raw_score=True)


y1 = [1 if i=="Intimacy" else 0 for i in tpr_cat.iloc[:,11]]  # change 6, 11
y2 = [1 if i=="Intimacy" else 0 for i in tpr_lgbm.iloc[:,11]]
y3 = [1 if i=="Intimacy" else 0 for i in tpr_rf.iloc[:,11]]

mean_fpr = np.linspace(0, 1, 100)

fpr, tpr, thresholds = roc_curve(y1, tpr_cat.iloc[:,13])  # change 8, 13
cat_tpr =  np.interp(mean_fpr, fpr, tpr)
fpr, tpr, thresholds = roc_curve(y2, tpr_lgbm.iloc[:,13])
lgbm_tpr =  np.interp(mean_fpr, fpr, tpr)
fpr, tpr, thresholds = roc_curve(y3, tpr_rf.iloc[:,13])
rf_tpr =  np.interp(mean_fpr, fpr, tpr)


np.save("1.integrate_cat", cat_tpr)
np.save("1.integrate_lgbm",lgbm_tpr)
np.save("1.integrate_rf", rf_tpr)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.9370,0.9862,0.9370,0.9374,0.9369,0.8734,0.8739


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.9486,0.9899,0.9486,0.9487,0.9486,0.8967,0.8969


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.9486,0.9858,0.9486,0.9487,0.9486,0.8967,0.8969
